# 实际利率来源与配对Dataset资格

本卡只验证来源，不进行预测或策略实验。机制依据见plan.json的官方FRED/H.15及Fed研究链接；利率渠道与未来半导体超额的关系只是待检验推断。固定20-session差分，未做相关性、期限、符号或阈值搜索。

In [ ]:
import os, sys, json, hashlib
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
ENTRY_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "AGENTS.md").is_file())
EVIDENCE_ROOT = Path(os.environ.get("SOXL_EVIDENCE_ROOT", ENTRY_ROOT)).resolve()
SOURCE_ROOT = Path(os.environ.get("SOXL_RESEARCH_SOURCE_ROOT", ENTRY_ROOT)).resolve()
sys.path.insert(0,str(SOURCE_ROOT))
CARD = ENTRY_ROOT / "research" / "real_yield_source_qualification_20261012"
PLAN = json.loads((CARD / "plan.json").read_text())
PLAN_BYTES = (CARD / "plan.json").read_bytes()
for relative,digest in PLAN["raw_files_sha256"].items():
    assert hashlib.sha256((EVIDENCE_ROOT/relative).read_bytes()).hexdigest()==digest
for relative,digest in PLAN["source_code_sha256"].items():
    assert hashlib.sha256((SOURCE_ROOT/relative).read_bytes()).hexdigest()==digest
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.provider_content_hash import provider_content_hash
from src.data.alfred_yield_information import read_initial_yield_archive, qualify_yield_change, build_yield_information_store
price_provider = EVIDENCE_ROOT / PLAN["price_provider_uri"]
assert provider_content_hash(price_provider)[0] == PLAN["price_content_sha256"]
qlib.init(provider_uri=str(price_provider),region="us",kernels=1)
calendar = pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time=PLAN["as_of"],freq="day"))
raw = read_initial_yield_archive(EVIDENCE_ROOT/PLAN["raw_root"]/PLAN["series_id"],as_of=PLAN["as_of"],series_id=PLAN["series_id"])
state, events, audit = qualify_yield_change(raw,calendar,change_sessions=PLAN["change_sessions"])
print(audit)


In [ ]:
provider = EVIDENCE_ROOT / PLAN["information_provider_uri"]
manifest,native,provider_hash = build_yield_information_store(state,provider_uri=provider,as_of=PLAN["as_of"],source="official_alfred_dfii10_initial_release_20261012_v1",instrument=PLAN["information_instrument"])
assert provider_content_hash(price_provider)[0] == PLAN["price_content_sha256"]
from src.data.oos_horizon_inputs import prepare_horizon_inputs
from src.data.information_increment import load_information_state,build_matched_information_datasets
from src.model.generic_annual_ridge_selection import qualify_annual_horizon_inputs
from src.data.convex_allocation import LABEL_COLUMNS
price_plan=json.loads((EVIDENCE_ROOT/"research/purged_ridge_regularization_20261009/plan.json").read_text())
price=prepare_horizon_inputs(EVIDENCE_ROOT,price_plan)
information=load_information_state(provider_uri=provider,content_sha256=provider_hash,instrument=PLAN["information_instrument"],field="$real_yield_change",signal_dates=price["frame"].index)
bundle=build_matched_information_datasets(price,information,as_of=PLAN["as_of"])
assert bundle["fit_performed"] is False
assert len(bundle["control_features"])==25 and len(bundle["candidate_features"])==26
assert bundle["frame"].index.equals(calendar)
market_positions=bundle["calendar"].get_indexer(calendar)
assert (market_positions>=0).all() and all(b-a==1 for a,b in zip(market_positions[:-1],market_positions[1:]))
print({"signal_calendar_rows":len(calendar),"full_market_calendar_rows":len(bundle["calendar"]),"signal_start":str(calendar[0].date()),"market_start":str(bundle["calendar"][0].date()),"full_market_calendar_retained":True})
eligibility=[]
for year in range(2016,2027):
    arms=[]
    for arm in ["control","candidate"]:
        q=qualify_annual_horizon_inputs(bundle[arm],bundle["frame"],outer_year=year,horizon_sessions=5,feature_columns=bundle[arm+"_features"],label_columns=LABEL_COLUMNS,availability_column="h5_available_at",eligibility_column="source_known")
        arms.append(q["audit"])
    for key in ["inner_identity","outer_identity","validation_identity"]:
        assert arms[0][key]==arms[1][key]
    eligibility.append(dict(outer_year=year,inner_training_rows=arms[0]["inner_training_rows"],validation_rows=arms[0]["validation_rows"],outer_training_rows=arms[0]["outer_training_rows"],matched_identity=True,source_excluded_inner=arms[0]["inner_source_excluded_rows"]))
eligibility=pd.DataFrame(eligibility)
display(eligibility)


In [ ]:
from src.core.recorder_run import start_evidence_run,archive_evidence,publish_evidence_manifest
identity=start_evidence_run(experiment=PLAN["experiment"],uri=CARD/"source_recorder",name="yield_source_qualification",objects={"plan.pkl":PLAN,"raw.pkl":raw,"source_events.pkl":events,"source_state.pkl":state,"source_readback.pkl":native,"source_audit.pkl":audit,"matched_inputs.pkl":bundle,"eligibility.pkl":eligibility},params={"models_trained":0,"strategy_replays":0,"source_series":PLAN["series_id"]})
policy=dict(policy="slim_card_recorder_first",workflow_config="workflow.yaml",recorder_only_artifacts=["inputs/plan.pkl","inputs/raw.pkl","inputs/source_events.pkl","inputs/source_state.pkl","inputs/source_readback.pkl","inputs/source_audit.pkl","inputs/matched_inputs.pkl","inputs/eligibility.pkl"])
summary=dict(identity,status="source_and_matched_dataset_qualified_predictive_value_untested",recorder_status="FINISHED",models_trained=0,strategy_replays=0,orders=0,promotion_eligible=False,source_deployment_gate=False,as_of=PLAN["as_of"],region="us",source={"provider_uri":PLAN["information_provider_uri"],"price_provider_uri":PLAN["price_provider_uri"],"raw_root":PLAN["raw_root"]},provider_content_sha256=provider_hash,source_audit=audit,matched_dataset={"rows":bundle["rows"],"known_rows":bundle["known_rows"],"unknown_rows":bundle["unknown_rows"],"control_features":25,"candidate_features":26,"eligible_outer_years":eligibility.outer_year.tolist(),"inner_fit_performed":False,"signal_calendar_rows":len(calendar),"full_market_calendar_rows":len(bundle["calendar"]),"full_market_calendar_retained":True},plan_sha256=hashlib.sha256(PLAN_BYTES).hexdigest(),artifact_policy=policy,qlib_docs=["https://qlib.readthedocs.io/en/latest/component/data.html","https://qlib.readthedocs.io/en/latest/component/recorder.html"])
assert (CARD/"plan.json").read_bytes()==PLAN_BYTES
assert provider_content_hash(price_provider)[0]==PLAN["price_content_sha256"]
archive_evidence(identity,{"final_summary.pkl":summary})
publish_evidence_manifest(CARD,summary)
eligibility.to_csv(CARD/"source_eligibility.csv",index=False)
events.loc[events.premature_release].to_csv(CARD/"source_anomalies.csv",index=False)
import yaml
(CARD/"workflow.yaml").write_text(yaml.safe_dump(dict(experiment=PLAN["experiment"],source=summary["source"],recorder_id=summary["recorder_id"],artifact_policy=policy),sort_keys=False))
(CARD/"selected_model.json").write_text(json.dumps(dict(selection="none_source_qualification_only",models_trained=0,promotion_decision="not_promoted"),indent=2)+"\n")
conclusion=f"""# 实际利率初次发布来源资格

ALFRED DFII10原始初次发布档案已读取并经Qlib原生store、Data API、配对DatasetH验证。原始观察{audit['raw_rows']}条；共同日历{audit['calendar_rows']}日；20-session差分已知{audit['known_signal_rows']}日，未知{audit['unknown_signal_rows']}日。负实际利率观测{audit['negative_yield_observations']}条，原样保留。

来源值只在观察日与初次发布日期两者之后的下一真实交易日09:31纽约时间可用。未知最新观察不填旧值；非交易日与过早发布日期记录保留在审计。日期级vintage可核验，秒级首次公开未认证，source_deployment_gate=false。

2016—2026各年度的25特征控制组和26特征候选组在严格成熟训练/验证样本身份上相同。资格通过不等于信息增量，更不等于稳定超额。模型拟合0、策略回放0、订单0。

晋升决定：不晋升；预测价值未测试。下一步已冻结单个real_yield_change20增量的范围：H5、10/25bp、四个固定prior/Ridge候选、最多110次拟合和12个原生场景；完整prefit与执行计划尚需先登记，然后才允许执行。未开启模型实验。20日差分是待检验的机制假设，不能从来源研究推断预测有效。历史为复用诊断；成功仍需60个独立合格前向日。

Recorder {summary['recorder_id']}；FINISHED。
"""
(CARD/"conclusion.md").write_text(conclusion)
display(Markdown(conclusion))
print({"source_recorder":summary["recorder_id"],"provider_hash":provider_hash,"new_fits":0,"matched_annual_cohorts":len(eligibility)})


中文结论在执行后从conclusion.md同步。